# 02 — PhoWhisper-small: zero-shot, LoRA fine-tuning, evaluation

Self-contained. Before running: *Settings → Accelerator: GPU T4*, *Internet: on*, and
*Add Input* → the shared dataset **`asr-noisy-vi`** (built by `01_prepare_dataset`: clean
speech, noise pools, noisy test / dev files, manifests).

| Step | What | Output in `/kaggle/working` |
|---|---|---|
| 0 | Configuration | |
| 1 | Clone the repo, install it | |
| 2 | Find the dataset, load the test / dev tables (9 conditions: clean, traffic and babble at 20/10/5/0 dB) | |
| 3 | Load PhoWhisper-small, `transcribe(model, processor, audios)`, shared helpers | |
| 4 | Sanity check on 10 files, then the full zero-shot test | `predictions_phowhisper-small_zero-shot.csv` |
| 5 | Evaluate: WER / CER per noise × SNR, bootstrap intervals | |
| 6 | Fine-tuning setup: training set (online noise from `train_schedule.csv`), collate, LoRA | |
| 7 | Fine-tuning, dev WER after each epoch, keep the best adapter | `lora_*/`, `history_*.csv` |
| 8 | Test the best adapter, compare with zero-shot | `predictions_phowhisper-small_<variant>.csv` |

After step 2 the code only sees `(rows, audios)`: 16 kHz float32 arrays and normalized
reference text. Steps 6–8 can run in a later session: run steps 0–3, then jump to 6.

## 0. Configuration

In [ ]:
from pathlib import Path

# Code (only core / text / scoring helpers are used)
REPO_URL = "https://github.com/anhvinhphan659/asr-noisy-vi"
COMMIT = "83fad8741d1cb742f26250a1f23dd6880c764b60"
CODE_DIR = Path("/tmp/asr-noisy-vi")

# Model and decoding (fixed for every experiment)
MODEL = "phowhisper-small"
MODEL_ID = "vinai/PhoWhisper-small"
GENERATION_CONFIG_FALLBACK = "openai/whisper-small"   # same tokenizer; used if the checkpoint lacks a language table
LANGUAGE = "vi"
NUM_BEAMS = 1                # greedy
MAX_NEW_TOKENS = 400
EVAL_BATCH = 16

# Fine-tuning
VARIANT = "ft-noisy"         # "ft-noisy" (online noise from train_schedule.csv) or "ft-clean"
EPOCHS = 5                   # <= 10, the length of train_schedule.csv
TRAIN_BATCH = 16
LR = 2e-4
WARMUP_FRACTION = 0.1
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05
LORA_TARGETS = ["q_proj", "v_proj"]
MAX_LABEL_TOKENS = 448
NUM_WORKERS = 2
SEED = 42

SR = 16000
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)

## 1. Code: clone and install

If the repo is private, add a Kaggle secret `GITHUB_TOKEN` (*Add-ons → Secrets*).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _token = None
_url = REPO_URL.replace("https://", f"https://{_token}@") if _token else REPO_URL

!rm -rf {CODE_DIR}
!git clone -q {_url} {CODE_DIR}
!git -C {CODE_DIR} checkout -q {COMMIT}
!git -C {CODE_DIR} log -1 --format="%h %ad %s" --date=short
!pip install -q -e "{CODE_DIR}[notebooks]" peft
!pip uninstall -y -q torchao      # Kaggle's torchao is too old for recent peft; not used here

In [ ]:
import sys
sys.path.insert(0, str(CODE_DIR))      # importable in this kernel without a restart

import gc
import math
import re

import numpy as np
import pandas as pd
import torch
from IPython.display import Audio, display
from tqdm.auto import tqdm

from core import load_audio
from dataprep import (collate_raw, describe, find_data_root, iter_eval_batches, load_eval_set,
                      load_train_set, require_data)
from evaluation import PREDICTION_COLUMNS, bootstrap_ci, compare, score, summarize

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
torch.manual_seed(SEED)
print(DEVICE, torch.__version__)

## 2. Data

Cell 1 finds the shared dataset (the folder holding `DATASET.json`, wherever Kaggle mounts
it). Cell 2 picks what to test (uncomment a line) and builds `test_set` / `dev_set`: one
row per file, with `path` (noisy or clean audio) and `text` (normalized reference). The
next steps only use `test_set`, `dev_set` and `iter_batches`.

In [ ]:
# 2.1 Find the dataset
!find /kaggle/input -maxdepth 6 -name DATASET.json 2>/dev/null

DATA_ROOT = None                            # or set the folder holding DATASET.json by hand
for depth in range(1, 7):
    hits = sorted(Path("/kaggle/input").glob("/".join(["*"] * depth) + "/DATASET.json"))
    if DATA_ROOT is None and hits:
        DATA_ROOT = hits[0].parent
if DATA_ROOT is None:
    DATA_ROOT = find_data_root()            # $ASR_DATA_ROOT, ./asr-data; raises with instructions

INFO = require_data(DATA_ROOT)              # stops with a clear message if something is missing
M = DATA_ROOT / "manifests"
clean = pd.read_csv(M / "clean.csv")
print(DATA_ROOT)
print(describe(INFO))                       # keep this line with the results

In [ ]:
# 2.2 Pick what to test: uncomment one line in each group
DATASETS = ["fleurs"]
# DATASETS = ["vivos"]
# DATASETS = ["vietmed"]
# DATASETS = ["fleurs", "vivos", "vietmed"]

CONDITIONS = None                                            # all 9 conditions
# CONDITIONS = ["clean"]
# CONDITIONS = ["clean", "traffic_5dB", "babble_5dB"]
# CONDITIONS = ["clean", "traffic_20dB", "traffic_10dB", "traffic_5dB", "traffic_0dB"]
# CONDITIONS = ["clean", "babble_20dB", "babble_10dB", "babble_5dB", "babble_0dB"]

LIMIT = None                                                 # all utterances
# LIMIT = 20                                                 # quick run: 20 utterances per dataset

test_set = load_eval_set(M / "test_plan.csv", clean, datasets=DATASETS, conditions=CONDITIONS, limit=LIMIT)
dev_set = load_eval_set(M / "dev_plan.csv", clean, datasets=DATASETS, limit=LIMIT)
iter_batches = lambda table, batch_size: iter_eval_batches(table, DATA_ROOT, batch_size=batch_size)

print(f"test: {len(test_set)} files, {test_set['utt_id'].nunique()} utterances, "
      f"{test_set['duration_s'].sum() / 3600:.1f} h | dev: {len(dev_set)} files")
display(test_set.groupby(["dataset", "condition"]).size().unstack(0))
row = test_set.iloc[-1]
print(row["condition"], "|", row["text"])
Audio(str(DATA_ROOT / row["path"]))

## 3. Model and `transcribe`

Decoding is fixed for every experiment: Vietnamese, transcribe task, greedy
(`num_beams=1`, no sampling), at most `MAX_NEW_TOKENS` tokens. The output is returned as the
model writes it; scoring normalizes both sides.

In [ ]:
from transformers import GenerationConfig, WhisperForConditionalGeneration, WhisperProcessor

DECODE = dict(language=LANGUAGE, task="transcribe", num_beams=NUM_BEAMS, do_sample=False,
              max_new_tokens=MAX_NEW_TOKENS)

processor = WhisperProcessor.from_pretrained(MODEL_ID)


def load_base_model():
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
    if getattr(model.generation_config, "lang_to_id", None) is None:
        # Some fine-tuned checkpoints lack the language table that generate(language=...) needs.
        model.generation_config = GenerationConfig.from_pretrained(GENERATION_CONFIG_FALLBACK)
    model.generation_config.forced_decoder_ids = None
    model.generation_config.max_length = None          # use only max_new_tokens
    model.config.forced_decoder_ids = None
    return model.to(DEVICE).eval()


@torch.no_grad()
def transcribe(model, processor, audios):
    # list of 16 kHz float32 arrays -> list of raw transcripts
    features = processor.feature_extractor(audios, sampling_rate=SR,
                                           return_tensors="pt").input_features.to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
        ids = model.generate(input_features=features, **DECODE)
    return [text.strip() for text in processor.batch_decode(ids, skip_special_tokens=True)]


model = load_base_model()
print(transcribe(model, processor, [load_audio(DATA_ROOT / test_set["path"].iloc[0])]))

### Shared helpers: run a model on a table, evaluate predictions

`evaluate` reuses `evaluation.score` (same `normalize_text` on reference and hypothesis) and
`summarize` (corpus-level WER / CER in percent, split into substitutions, deletions,
insertions).

In [ ]:
VI_DIACRITICS = re.compile("[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]",
                           re.IGNORECASE)
# Latin letters with Vietnamese diacritics, digits, common punctuation
ALLOWED = re.compile(r"[\s0-9A-Za-zÀ-ɏ̀-ͯḀ-ỿ.,!?;:'\"()%&/\-–…]")
SNR_ORDER = ["clean", "20 dB", "10 dB", "5 dB", "0 dB"]


def run_predictions(model, table, variant, batch_size=EVAL_BATCH, desc=None):
    rows, hyps = [], []
    for batch_rows, audios in tqdm(iter_batches(table, batch_size),
                                   total=math.ceil(len(table) / batch_size), desc=desc or variant):
        rows.append(batch_rows)
        hyps += transcribe(model, processor, audios)
    out = pd.concat(rows)[["utt_id", "condition", "dataset", "noise_type", "snr_db", "text"]]
    return out.assign(model=MODEL, variant=variant, hyp=hyps).reset_index(drop=True)


def predictions_path(variant):
    return OUT / f"predictions_{MODEL}_{variant}.csv"


def run_or_load(model, table, variant):
    # Reuse the saved CSV when it covers exactly this table; otherwise run and save.
    path = predictions_path(variant)
    if path.exists():
        saved = pd.read_csv(path, dtype={"hyp": str}, keep_default_na=False)
        if set(zip(saved["utt_id"], saved["condition"])) == set(zip(table["utt_id"], table["condition"])):
            print(f"loaded {path}")
            return saved
    pred = run_predictions(model, table, variant)
    pred.to_csv(path, index=False)
    print(f"wrote {path} ({len(pred)} rows)")
    return pred


def check_outputs(pred):
    report = pred[["utt_id", "condition", "text", "hyp"]].copy()
    report["diacritics"] = report["hyp"].str.contains(VI_DIACRITICS)
    report["odd_chars"] = report["hyp"].map(lambda t: "".join(sorted({c for c in t if not ALLOWED.match(c)})))
    report["special_token"] = report["hyp"].str.contains("<|", regex=False)
    report["empty"] = report["hyp"].str.strip().eq("")
    return report


def evaluate(pred, table, by=("model", "variant", "dataset", "noise_type", "snr_db")):
    plan = table[["utt_id", "condition", "dataset", "noise_type", "snr_db"]]
    refs = table[["utt_id", "text"]].drop_duplicates("utt_id")
    scored = score(pred[PREDICTION_COLUMNS], plan, refs)
    return scored, summarize(scored, by=list(by))


def wer_grid(summary, metric="wer"):
    # One row per system × noise type, one column per SNR.
    s = summary.assign(snr=summary["snr_db"].map(lambda v: "clean" if pd.isna(v) else f"{v:g} dB"))
    grid = s.pivot_table(index=["model", "variant", "dataset", "noise_type"], columns="snr",
                         values=metric, aggfunc="first")
    return grid[[c for c in SNR_ORDER if c in grid.columns]]


def worst(scored, n=5):
    errors = scored[["w_sub", "w_del", "w_ins"]].sum(axis=1) / scored["w_n_ref"].clip(lower=1)
    return scored.assign(utt_wer=(100 * errors).round(1)).nlargest(n, "utt_wer")[
        ["utt_id", "condition", "utt_wer", "ref", "hyp_norm"]]

## 4. Zero-shot test

Sanity check on 10 clean files (read every transcript), then the full test table.

In [ ]:
sample = test_set[test_set["condition"] == "clean"]
sample = (sample if len(sample) else test_set).drop_duplicates("utt_id").head(10)
report = check_outputs(run_predictions(model, sample, "zero-shot", desc="sanity"))
for r in report.itertuples():
    print(f"[{r.utt_id}]\n  ref: {r.text}\n  hyp: {r.hyp}\n")
display(report[["utt_id", "diacritics", "odd_chars", "special_token", "empty"]])
problems = report[~report["diacritics"] | report["odd_chars"].ne("") | report["special_token"] | report["empty"]]
print("OK" if problems.empty else f"CHECK {len(problems)} transcripts above before the full run")

In [ ]:
pred_zs = run_or_load(model, test_set, "zero-shot")
pred_zs.sample(5, random_state=SEED)[["condition", "text", "hyp"]]

## 5. Evaluate

WER / CER per noise type × SNR, split into substitutions, deletions and insertions;
95% bootstrap intervals; the worst utterances.

In [ ]:
scored_zs, summary_zs = evaluate(pred_zs, test_set)
print("WER (%)"); display(wer_grid(summary_zs))
print("CER (%)"); display(wer_grid(summary_zs, "cer"))
display(summary_zs[["noise_type", "snr_db", "wer", "cer", "sub_pct", "del_pct", "ins_pct"]])
display(bootstrap_ci(scored_zs))
display(worst(scored_zs))

## 6. Fine-tuning setup

`load_train_set` serves the train utterances of `DATASETS`. `ft-noisy` mixes traffic or
babble from the train noise pool on the fly, from the row of `train_schedule.csv` for the
current epoch (25% clean, SNR 0–20 dB): each epoch gets new noise and every model gets the
same samples. `ft-clean` serves clean audio. LoRA is added to `q_proj` / `v_proj` of the
encoder and decoder; the base weights are frozen.

In [ ]:
# Kaggle ships an old torchao that recent peft rejects; we do not use torchao.
import peft.import_utils
import peft.tuners.lora.torchao
peft.import_utils.is_torchao_available = lambda: False
peft.tuners.lora.torchao.is_torchao_available = lambda: False
from peft import LoraConfig, PeftModel, get_peft_model
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup

train_set = load_train_set(M / "clean.csv", DATA_ROOT, VARIANT, schedule=M / "train_schedule.csv",
                           noise_pools=[M / "noise_traffic.csv", M / "noise_voices.csv"],
                           datasets=DATASETS)
print(f"{VARIANT}: {len(train_set)} training utterances")

tokenizer = processor.tokenizer
tokenizer.set_prefix_tokens(language="vietnamese", task="transcribe", predict_timestamps=False)

for name in ["model", "ft_model", "best_model"]:          # free GPU memory from earlier steps
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

base = load_base_model()
DECODER_START = base.config.decoder_start_token_id
ft_model = get_peft_model(base, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                           target_modules=LORA_TARGETS, bias="none"))
ft_model.print_trainable_parameters()


def collate(items):
    batch = collate_raw(items)             # utt_id, audio, text, noise_type, snr_db (lists)
    features = processor.feature_extractor(batch["audio"], sampling_rate=SR,
                                           return_tensors="pt").input_features
    tokens = tokenizer(batch["text"], padding=True, truncation=True, max_length=MAX_LABEL_TOKENS,
                       return_tensors="pt")
    labels = tokens.input_ids.masked_fill(tokens.attention_mask.ne(1), -100)   # ignore padding
    if (labels[:, 0] == DECODER_START).all():
        labels = labels[:, 1:]             # the model prepends the start token itself
    return features, labels

In [ ]:
# Check one batch and listen to one training sample
train_set.set_epoch(1)
item = train_set[0]
print(item["noise_type"], item["snr_db"], "|", item["text"])
display(Audio(item["audio"], rate=SR))

features, labels = collate([train_set[i] for i in range(4)])
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
    loss = ft_model(input_features=features.to(DEVICE), labels=labels.to(DEVICE)).loss
print(features.shape, labels.shape, f"loss {loss.item():.3f}")
print(tokenizer.decode(labels[0][labels[0] != -100]))

## 7. Fine-tuning

Dev WER is measured before training (epoch 0 = zero-shot) and after each epoch. The adapter
with the lowest dev WER is saved. The test set is not used here.

In [ ]:
def dev_wer(model):
    pred = run_predictions(model, dev_set, VARIANT, desc="dev")
    _, summary = evaluate(pred, dev_set, by=("model", "variant"))
    return float(summary["wer"].iloc[0])


ADAPTER_DIR = OUT / f"lora_{MODEL}_{VARIANT}"
steps_per_epoch = math.ceil(len(train_set) / TRAIN_BATCH)
optimizer = torch.optim.AdamW([p for p in ft_model.parameters() if p.requires_grad], lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRACTION * EPOCHS * steps_per_epoch),
                                            EPOCHS * steps_per_epoch)
scaler = torch.amp.GradScaler("cuda", enabled=AMP)

ft_model.eval()
history = [{"epoch": 0, "train_loss": None, "dev_wer": dev_wer(ft_model)}]
print(history[-1])
best_wer = math.inf

for epoch in range(1, EPOCHS + 1):
    train_set.set_epoch(epoch)             # new noise for this epoch, the same for every model
    loader = DataLoader(train_set, batch_size=TRAIN_BATCH, shuffle=True, collate_fn=collate,
                        num_workers=NUM_WORKERS, generator=torch.Generator().manual_seed(SEED + epoch))
    ft_model.train()
    losses = []
    for features, labels in tqdm(loader, desc=f"epoch {epoch}"):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            loss = ft_model(input_features=features.to(DEVICE), labels=labels.to(DEVICE)).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        losses.append(loss.item())
    ft_model.eval()
    wer = dev_wer(ft_model)
    history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "dev_wer": wer})
    if wer < best_wer:
        best_wer = wer
        ft_model.save_pretrained(ADAPTER_DIR)
    print(history[-1], "(best)" if wer == best_wer else "")
    pd.DataFrame(history).to_csv(OUT / f"history_{MODEL}_{VARIANT}.csv", index=False)

pd.DataFrame(history)

## 8. Test the fine-tuned model

Loads the best adapter, runs the same test table with the same decoding, and compares it
with zero-shot. `compare` is a paired bootstrap of `WER(fine-tuned) − WER(zero-shot)`: a
negative `diff` means fewer errors; `significant` is True when the 95% interval excludes 0.

In [ ]:
for name in ["model", "ft_model", "base"]:
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

best_model = PeftModel.from_pretrained(load_base_model(), ADAPTER_DIR).to(DEVICE).eval()
pred_ft = run_or_load(best_model, test_set, VARIANT)
pred_zs = run_or_load(load_base_model() if not predictions_path("zero-shot").exists() else None,
                      test_set, "zero-shot")
scored_ft, _ = evaluate(pred_ft, test_set)
scored_zs, _ = evaluate(pred_zs, test_set)

both = pd.concat([scored_zs, scored_ft], ignore_index=True)
print("WER (%)"); display(wer_grid(summarize(both)))
print("CER (%)"); display(wer_grid(summarize(both), "cer"))
display(compare(both, (MODEL, "zero-shot"), (MODEL, VARIANT)))
display(worst(scored_ft))